# Question Classification

## 1. Data preparation

In [9]:
import json
import pandas as pd

def read_json_to_dataframe(file_path: str, orient: str = 'records') -> pd.DataFrame:
    try:
        df = pd.read_json(file_path, orient=orient)
        print(f"Successfully read JSON file: {file_path}")
        return df
    except Exception as e:
        print(f"Error reading JSON file: {e}")
        return pd.DataFrame()

In [11]:
input_path = "/kaggle/input/question-tags/question.json"
df = read_json_to_dataframe(input_path)
tag_counter = Counter(chain.from_iterable(df["tags"]))
tag_counts = pd.Series(tag_counter).sort_values(ascending=False)
print("Number of subject: ", len(tag_counts))
print("Top 10 tags:", tag_counts.head(10).to_dict())
print("Bottom 10 tags:", tag_counts.tail(10).to_dict())

Successfully read JSON file: /kaggle/input/question-tags/question.json
Number of subject:  51
Top 10 tags: {'c#': 52305, '.net': 28379, 'java': 27490, 'asp.net': 23288, 'php': 21658, 'javascript': 20841, 'c++': 18313, 'jquery': 14928, 'python': 14480, 'sql': 13508}
Bottom 10 tags: {'cocoa-touch': 3417, 'oracle': 3285, 'cocoa': 3234, 'algorithm': 3233, 'svn': 3219, 'arrays': 3202, 'security': 3145, 'perl': 3108, 'sharepoint': 3091, 'delphi': 3091}


## 2. Logistic Regression (One-vs-Rest)

### 2.1. Import library

In [12]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score, classification_report, precision_score, recall_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import hamming_loss
import joblib
import numpy as np

### 2.2. Convert labels to binary format (multi-label)

- Filter out rare labels

In [13]:
mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(df["tags"])
print(Y.shape)

(308071, 51)


### 2.3. Vectorize text using TF-IDF

In [15]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),    # unigram + bigram
    sublinear_tf=True,
    min_df=2
)

X = vectorizer.fit_transform(df["text"])
print(X.shape)

(308071, 1771208)


### 2.4. Split the dataset into training and test sets

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

### 2.5. Train a Logistic Regression model using the One-vs-Rest strategy

In [18]:
log_reg = LogisticRegression(
    max_iter=1000,
    penalty='l2',
    C=1.0,
    tol=1e-4,
    n_jobs=-1
)

model = OneVsRestClassifier(log_reg)
model.fit(X_train, y_train)

OneVsRestClassifier(estimator=LogisticRegression(max_iter=1000, n_jobs=-1))

### 2.6. Model evaluation

In [22]:
# Dữ liệu đầu vào
text = ["Given a select with multiple option's in jQuery. How should the key be stored and retrieved? The ID may be an OK place but would not be guaranteed unique if I had multiple select's sharing values (and other scenarios). Thanks and in the spirit of TMTOWTDI. $select = $(\"<select><\/select>\");\n$select.append(\"<option>Jason<\/option>\") \/\/Key = 1\n       .append(\"<option>John<\/option>\") \/\/Key = 32\n       .append(\"<option>Paul<\/option>\") \/\/Key = 423\n$option = $(\"<option><\/option>\");\n$select = $(\"<select><\/select>\");\n$select.addOption = function(value,text){\n    $(this).append($(\"<option\/>\").val(value).text(text));\n};\n\n$select.append($option.val(1).text(\"Jason\").clone())\n       .append(\"<option value=32>John<\/option>\")\n       .append($(\"<option\/>\").val(423).text(\"Paul\"))\n       .addOption(\"321\",\"Lenny\");"]
    
X_test_new = vectorizer.transform(text)
y_pred_proba = model.predict_proba(X_test_new)[0]  # xác suất cho câu đầu tiên

# Tạo danh sách (label, probability)
labels_probs = list(zip(mlb.classes_, y_pred_proba))

# Sắp xếp theo xác suất từ cao xuống thấp
labels_probs_sorted = sorted(labels_probs, key=lambda x: x[1], reverse=True)

# Lấy top 3 nhãn
top3_labels = labels_probs_sorted[:3]

print("Top 3 predicted labels with probability:")
for label, prob in top3_labels:
    print(f"{label}: {prob:.4f}")

Top 3 predicted labels with probability:
jquery: 0.8244
javascript: 0.1808
sql: 0.0663


### 2.7. Save model and MultiLabelBinarizer

In [23]:
from joblib import dump, load
dump(model, 'log_reg_model.joblib')
dump(vectorizer, 'vectorizer.joblib')
dump(mlb, 'mlb.joblib')
print("Saved: log_reg_model.joblib, vectorizer.joblib, mlb.joblib")

Saved: log_reg_model.joblib, vectorizer.joblib, mlb.joblib
